# Overview

I'm creating a notebook to process MOM6 data in order to recreate the Sea Surface Temperature (SST) anomaly plot that Isaac Schroeder produced for the 2025 mini ESR.

Currently, the plot is produced with [OISST](https://www.ncei.noaa.gov/products/optimum-interpolation-sst). Here's a brief description of that dataset: "The Optimum Interpolation Sea Surface Temperature (OISST) dataset is a global sea surface temperature dataset. Data is on a regular grid at ¼ degree intervals, from September 1, 1981 to present. Temperature data comes from satellites, ships, buoys and Argo floats; temperature is interpolated in between observations."

Here's an example of the plot, for Spring 2025: [oc_miniESR_sst_mons_4_6_contour.png](https://github.com/id-sch/oc_cciea.github.io/blob/main/figures_gha/miniESR/2025/sst_anom/oc_miniESR_sst_mons_4_6_contour.png). The figure is plotted for 32-48 °N, 128-116.8 °W, at 1/2 degree resolution. The anomalies are calculated using a climatological mean calculated from monthly data 1982 - current, and subtracting that mean from seasonal mean (e.g. April to June, 2025 for spring 2025).

Here's the repository from Isaac: https://github.com/id-sch/oc_cciea.github.io. There are two key scripts he uses to produce the plot.
1. [sst_oi_daily_v21_update.py](https://github.com/id-sch/oc_cciea.github.io/blob/main/code_gha/SSTspatial/sst_oi_daily_v21_update.py) downloads OISST data every month using ERDDAP, and saves it in a netCDF.
2. [miniESR/countour_noaaoisst_cilm_anoms_season.py](https://github.com/id-sch/oc_cciea.github.io/blob/main/code_gha/miniESR/contour_noaaoisst_clim_anoms_season.py) calculates the climatological and seasonal means, calculates the anomalies, then plots the anomalies along the US West Coast.

I'm going to recreate the same plot using MOM6-NEP10k, a version of Modular Ocean Model 6 (MOM6) for the Northeast Pacific (NEP), available on the [CEFI data portal](https://psl.noaa.gov/cefi_portal/#data_access). The regirdded sea surface temperature data is stored in variable name *tos*. The most recent hindcast release is version *r20250912*, and has an available date range is 1993 to 2025. At time of writing, the forecast is not available on the portal for NEP. Instead of downloading from the THREDDS server, I'm going to use the demo [Earthmover platform](https://app.earthmover.io/NOAA-PMEL).

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import time
from arraylake import Client

In [2]:
client = Client()
repo = client.get_repo("NOAA-PMEL/cefi-nep-hindcast-monthly")
session = repo.readonly_session(branch="main")
aux = xr.open_zarr(session.store, zarr_format=3, group="regrid/aux1")
ds_temp = aux[["tos"]]
ds_temp.info()

xarray.Dataset {
dimensions:
	time = 396 ;
	lat = 815 ;
	lon = 341 ;

variables:
	float32 tos(time, lat, lon) ;
		tos:regrid_method = bilinear ;
		tos:units = degC ;
		tos:long_name = Sea Surface Temperature ;
		tos:cell_methods = area:mean jh:mean ih:mean time: mean ;
		tos:cell_measures = area: areacello ;
		tos:time_avg_info = average_T1,average_T2,average_DT ;
		tos:standard_name = sea_surface_temperature ;
	float64 lat(lat) ;
		lat:standard_name = latitude ;
		lat:long_name = latitude ;
		lat:units = degrees_north ;
		lat:axis = Y ;
		lat:actual_range = [10.809035301208496, 80.71794891357422] ;
	float64 lon(lon) ;
		lon:standard_name = longitude ;
		lon:long_name = longitude ;
		lon:units = degrees_east ;
		lon:axis = X ;
		lon:actual_range = [156.9248046875, 254.971923828125] ;
	datetime64[ns] time(time) ;
		time:long_name = time ;
		time:axis = T ;
		time:calendar_type = GREGORIAN ;
		time:bounds = time_bnds ;

// global attributes:
	:NumFilesInSet = 1 ;
	:title = NEP10k_202507_

In [3]:
ds_temp.coords

Coordinates:
  * time     (time) datetime64[ns] 3kB 1993-01-16T12:00:00 ... 2025-06-16
  * lat      (lat) float64 7kB 10.81 10.89 10.98 11.07 ... 80.55 80.63 80.72
  * lon      (lon) float64 3kB 156.9 157.2 157.5 157.8 ... 254.4 254.7 255.0

In [ ]:
# TODO - save a netCDF file with the relevant latitude, longitude, and temporal ranges
x_lim = [-128, -116.8]
y_lim = [32, 48]
ds_temp.sel(lat=slice(y_lim[1] + 360, y_lim[2] + 360), lon=slice(x_lim[1], x_lim[2]))
# check dataset size
print(ds_temp.nbytes / 1e9, "GB")

IndexError: list index out of range

In [ ]:
from dask.diagnostics import ProgressBar

# Wrap your save command in a ProgressBar context
with ProgressBar():
    ds_temp.to_netcdf("nep_temperature_subset.nc")

ModuleNotFoundError: No module named 'dask'